# Deployment Pipeline Notebook

This notebook consolidates your CRISP-DM flow into deployable training pipelines that include preprocessing, feature selection, and the three models used in your project.


In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import InstanceHardnessThreshold
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

RANDOM_STATE = 42
TARGET_COL = "y"
DATA_PATH = Path("data/data.csv")
ARTIFACTS_DIR = Path("models/deployment")


In [1]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.ensemble import RandomForestClassifier

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import InstanceHardnessThreshold


class ColumnDropper(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = list(columns)

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_df = X.copy()
        to_drop = [col for col in self.columns if col in X_df.columns]
        return X_df.drop(columns=to_drop)


class CrispDMPreprocessor(BaseEstimator, TransformerMixin):
    """Reproduces preprocessing steps from Preprocessment.ipynb, including cyclical month/day encoding."""

    def __init__(
        self,
        drop_columns=("month", "day"),
        log_column="previous",
        outlier_percentile=99.9,
        add_seasonal=True,
        month_col="month",
        day_col="day",
        drop_original_seasonal=True,
    ):
        self.drop_columns = tuple(drop_columns)
        self.log_column = log_column
        self.outlier_percentile = outlier_percentile
        self.add_seasonal = add_seasonal
        self.month_col = month_col
        self.day_col = day_col
        self.drop_original_seasonal = drop_original_seasonal

        self.month_map = {
            "jan": 1, "feb": 2, "mar": 3, "apr": 4,
            "may": 5, "jun": 6, "jul": 7, "aug": 8,
            "sep": 9, "oct": 10, "nov": 11, "dec": 12
        }

    def _apply_seasonal_encoding(self, X_df):
        X_out = X_df.copy()

        if not self.add_seasonal:
            return X_out

        month_exists = self.month_col in X_out.columns
        day_exists = self.day_col in X_out.columns

        if month_exists:
            month_series = X_out[self.month_col]

            if month_series.dtype == "object" or str(month_series.dtype).startswith("category"):
                month_series = month_series.astype(str).str.lower().map(self.month_map)

            month_series = pd.to_numeric(month_series, errors="coerce")

            X_out[f"{self.month_col}_sin"] = np.sin(2 * np.pi * month_series / 12)
            X_out[f"{self.month_col}_cos"] = np.cos(2 * np.pi * month_series / 12)

        if day_exists:
            day_series = pd.to_numeric(X_out[self.day_col], errors="coerce")

            X_out[f"{self.day_col}_sin"] = np.sin(2 * np.pi * day_series / 31)
            X_out[f"{self.day_col}_cos"] = np.cos(2 * np.pi * day_series / 31)

        if self.drop_original_seasonal:
            cols_to_drop = [col for col in [self.month_col, self.day_col] if col in X_out.columns]
            if cols_to_drop:
                X_out = X_out.drop(columns=cols_to_drop)

        return X_out

    def _prepare_base(self, X):
        X_df = X.copy()

        # 1) Add cyclical features first
        X_df = self._apply_seasonal_encoding(X_df)

        # 2) Then drop any extra columns requested by user
        cols = [col for col in self.drop_columns if col in X_df.columns]
        if cols:
            X_df = X_df.drop(columns=cols)

        return X_df

    def _apply_log(self, X_df):
        X_out = X_df.copy()
        if self.log_column in X_out.columns:
            X_out[self.log_column] = np.log1p(X_out[self.log_column])
        return X_out

    def _scale_numeric(self, X_df, fit=False):
        X_out = X_df.copy()
        if not self.numeric_cols_:
            return X_out

        num_data = X_out[self.numeric_cols_]
        if fit:
            std_scaled = self.std_scaler_.fit_transform(num_data)
            minmax_scaled = self.minmax_scaler_.fit_transform(std_scaled)
        else:
            std_scaled = self.std_scaler_.transform(num_data)
            minmax_scaled = self.minmax_scaler_.transform(std_scaled)

        X_out.loc[:, self.numeric_cols_] = minmax_scaled
        return X_out

    def _replace_unknown_with_nan(self, X_df):
        X_out = X_df.copy()
        for col in self.categorical_cols_:
            if col in X_out.columns:
                X_out[col] = X_out[col].replace("unknown", np.nan)
        return X_out

    def _fit_outlier_rules(self, X_df):
        self.outlier_rules_ = {}
        for col in self.numeric_cols_:
            series = pd.to_numeric(X_df[col], errors="coerce")
            mean_val = float(np.nanmean(series))
            std_val = float(np.nanstd(series))

            if np.isnan(std_val) or std_val == 0:
                self.outlier_rules_[col] = {"mean": mean_val, "std": std_val, "cutoff": np.inf}
                continue

            abs_z = np.abs((series - mean_val) / std_val)
            cutoff = float(np.nanpercentile(abs_z, self.outlier_percentile))
            self.outlier_rules_[col] = {"mean": mean_val, "std": std_val, "cutoff": cutoff}

    def _apply_outlier_rules(self, X_df):
        X_out = X_df.copy()
        for col, rule in self.outlier_rules_.items():
            std_val = rule["std"]
            cutoff = rule["cutoff"]
            if np.isnan(std_val) or std_val == 0 or np.isinf(cutoff):
                continue

            abs_z = np.abs((X_out[col] - rule["mean"]) / std_val)
            X_out.loc[abs_z > cutoff, col] = np.nan
        return X_out

    def _finalize_features(self, X_df, fit=False):
        if self.numeric_cols_:
            X_num = pd.DataFrame(
                self.num_imputer_.fit_transform(X_df[self.numeric_cols_]) if fit else self.num_imputer_.transform(X_df[self.numeric_cols_]),
                columns=self.numeric_cols_,
                index=X_df.index,
            )
        else:
            X_num = pd.DataFrame(index=X_df.index)

        if self.categorical_cols_:
            X_cat_imputed = pd.DataFrame(
                self.cat_imputer_.fit_transform(X_df[self.categorical_cols_]) if fit else self.cat_imputer_.transform(X_df[self.categorical_cols_]),
                columns=self.categorical_cols_,
                index=X_df.index,
            )

            if fit:
                X_cat_encoded = self.encoder_.fit_transform(X_cat_imputed)
            else:
                X_cat_encoded = self.encoder_.transform(X_cat_imputed)

            encoded_cols = self.encoder_.get_feature_names_out(self.categorical_cols_)
            X_cat = pd.DataFrame(X_cat_encoded, columns=encoded_cols, index=X_df.index)
        else:
            X_cat = pd.DataFrame(index=X_df.index)

        X_out = pd.concat([X_num, X_cat], axis=1)
        return X_out

    def fit(self, X, y=None):
        X_df = self._prepare_base(X)

        self.numeric_cols_ = X_df.select_dtypes(include=[np.number]).columns.tolist()
        self.categorical_cols_ = X_df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

        self.std_scaler_ = StandardScaler()
        self.minmax_scaler_ = MinMaxScaler()
        self.num_imputer_ = KNNImputer(n_neighbors=5, weights="distance")
        self.cat_imputer_ = SimpleImputer(strategy="most_frequent")
        self.encoder_ = OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop="first")

        X_work = self._apply_log(X_df)
        X_work = self._scale_numeric(X_work, fit=True)
        X_work = self._replace_unknown_with_nan(X_work)
        self._fit_outlier_rules(X_work)
        X_work = self._apply_outlier_rules(X_work)

        X_final = self._finalize_features(X_work, fit=True)
        self.feature_names_out_ = X_final.columns.tolist()
        return self

    def transform(self, X):
        X_df = self._prepare_base(X)
        X_work = self._apply_log(X_df)
        X_work = self._scale_numeric(X_work, fit=False)
        X_work = self._replace_unknown_with_nan(X_work)
        X_work = self._apply_outlier_rules(X_work)
        X_final = self._finalize_features(X_work, fit=False)
        return X_final

    def get_feature_names_out(self, input_features=None):
        return np.array(self.feature_names_out_)


class RandomForestTopKSelector(BaseEstimator, TransformerMixin):
    """Implements the RF top-k feature selection from FeatureSelection.ipynb."""

    def __init__(self, top_k=26, random_state=42):
        self.top_k = top_k
        self.random_state = random_state

    def fit(self, X, y):
        if hasattr(X, "columns"):
            self.feature_names_in_ = list(X.columns)
        else:
            self.feature_names_in_ = [f"f{i}" for i in range(X.shape[1])]

        self.selector_model_ = RandomForestClassifier(
            n_estimators=300,
            random_state=self.random_state,
            n_jobs=-1,
            class_weight="balanced",
            max_depth=5,
            max_features="sqrt",
        )

        self.selector_model_.fit(X, y)
        importances = self.selector_model_.feature_importances_

        k = min(self.top_k, len(importances))
        self.selected_indices_ = np.argsort(importances)[::-1][:k]
        self.selected_features_ = [self.feature_names_in_[i] for i in self.selected_indices_]
        return self

    def transform(self, X):
        if hasattr(X, "iloc"):
            return X.loc[:, self.selected_features_]
        return X[:, self.selected_indices_]

    def get_feature_names_out(self, input_features=None):
        return np.array(self.selected_features_)


def build_pipeline(model):
    return ImbPipeline(
        steps=[
            (
                "preprocess",
                CrispDMPreprocessor(
                    drop_columns=(),  # important: don't re-drop month/day here
                    log_column="previous",
                    outlier_percentile=99.9,
                    add_seasonal=True,
                    month_col="month",
                    day_col="day",
                    drop_original_seasonal=True,
                ),
            ),
            ("drop_duration", ColumnDropper(columns=["duration"])),
            ("undersample", InstanceHardnessThreshold(random_state=RANDOM_STATE)),
            ("feature_selection", RandomForestTopKSelector(top_k=26, random_state=RANDOM_STATE)),
            ("model", clone(model)),
        ]
    )

In [ ]:
df = pd.read_csv(DATA_PATH, sep=";")

X = df.drop(columns=[TARGET_COL]).copy()
y_raw = df[TARGET_COL].copy()

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"Classes: {dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))}")


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        random_state=RANDOM_STATE,
        max_iter=1000,
        class_weight="balanced",
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=RANDOM_STATE,
        class_weight="balanced",
        n_jobs=-1,
    ),
    "XGBoost": XGBClassifier(
        n_estimators=450,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
    ),
}

holdout_results = []
holdout_pipelines = {}

for model_name, estimator in models.items():
    pipeline = build_pipeline(estimator)
    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    if hasattr(pipeline, "predict_proba"):
        y_prob = pipeline.predict_proba(X_test)[:, 1]
    else:
        y_prob = y_pred

    holdout_results.append(
        {
            "model": model_name,
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "roc_auc": roc_auc_score(y_test, y_prob),
        }
    )

    holdout_pipelines[model_name] = pipeline

results_df = pd.DataFrame(holdout_results).sort_values("roc_auc", ascending=False).reset_index(drop=True)
champion_name = results_df.loc[0, "model"]

print("Holdout results:")
display(results_df)
print(f"Champion model: {champion_name}")


In [ ]:
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

full_data_pipelines = {}
for model_name, estimator in models.items():
    pipeline = build_pipeline(estimator)
    pipeline.fit(X, y)

    slug = model_name.lower().replace(" ", "_")
    joblib.dump(pipeline, ARTIFACTS_DIR / f"{slug}_pipeline.joblib")
    full_data_pipelines[model_name] = pipeline

champion_pipeline = full_data_pipelines[champion_name]

joblib.dump(champion_pipeline, ARTIFACTS_DIR / "champion_pipeline.joblib")
joblib.dump(label_encoder, ARTIFACTS_DIR / "label_encoder.joblib")

results_df.to_csv(ARTIFACTS_DIR / "holdout_metrics.csv", index=False)

selected_features = pd.DataFrame(
    {"feature": champion_pipeline.named_steps["feature_selection"].selected_features_}
)
selected_features.to_csv(ARTIFACTS_DIR / "champion_selected_features.csv", index=False)

print(f"Saved deployment artifacts to: {ARTIFACTS_DIR.resolve()}")
print("Files:")
for path in sorted(ARTIFACTS_DIR.glob("*")):
    print(f"- {path.name}")


In [ ]:
champion_loaded = joblib.load(ARTIFACTS_DIR / "champion_pipeline.joblib")
label_encoder_loaded = joblib.load(ARTIFACTS_DIR / "label_encoder.joblib")

inference_sample = df.drop(columns=[TARGET_COL]).head(5).copy()
pred_encoded = champion_loaded.predict(inference_sample).astype(int)
pred_labels = label_encoder_loaded.inverse_transform(pred_encoded)
pred_proba = champion_loaded.predict_proba(inference_sample)[:, 1]

preview = inference_sample.copy()
preview["predicted_class_encoded"] = pred_encoded
preview["predicted_class_label"] = pred_labels
preview["probability_class_1"] = pred_proba

preview
